# Telecom Customer Churn – Exploratory Analysis

Walkthrough of the analysis: data quality checks, cleaning, segment analysis in SQL, and an explainable churn-risk model.

*All data is synthetic.*

In [1]:
import sys, sqlite3
sys.path.append('../src')
import pandas as pd
from clean import clean
from analysis import load_queries, run_sql, fit_model
pd.set_option('display.width', 120)

## 1. Profile the raw extract
Before any analysis, check shape, nulls, duplicates, and category labels.

In [2]:
raw = pd.read_csv('../data/raw/customers.csv')
print(raw.shape)
raw.head()

(7570, 12)


,customer_id,region,plan,contract_type,payment_method,tenure_months,internet_speed_mbps,monthly_charges,support_tickets_90d,outages_90d,churned,total_charges
0,CUST-04917,Midwest,Internet Only,One year,Auto-pay (bank),44,500,51.69,0,2,No,2274.36
1,CUST-01243,North Texas,Internet + Mobile,Month-to-month,Auto-pay (bank),27,300,104.89,3,1,No,2832.03
2,CUST-00460,Northeast,Internet + TV,One year,Mailed check,29,100,106.45,3,1,No,3087.05
3,CUST-04594,North Texas,Internet Only,Month-to-month,Auto-pay (card),11,100,48.74,1,1,Yes,536.14
4,CUST-03665,North Texas,Triple Play,Month-to-month,Online one-time,14,300,125.73,0,0,No,1760.22


In [3]:
pd.DataFrame({'nulls': raw.isna().sum(), 'dtype': raw.dtypes.astype(str)})

,nulls,dtype
customer_id,0,str
region,0,str
plan,0,str
contract_type,0,str
payment_method,62,str
tenure_months,0,int64
internet_speed_mbps,0,int64
monthly_charges,95,float64
support_tickets_90d,0,int64
outages_90d,0,int64


In [4]:
print('Exact duplicate rows:', raw.duplicated().sum())
print('Distinct contract labels:', sorted(raw['contract_type'].unique()))

Exact duplicate rows: 70
Distinct contract labels: ['Month-to-month', 'One year', 'Two year', 'month to month', 'one year', 'two year']


## 2. Clean, with an audit log of every fix

In [5]:
df, log = clean(raw)
pd.DataFrame(log)

,step,rows_affected
0,Removed exact duplicate rows,70
1,Removed duplicate customer_id rows,0
2,Trimmed / re-cased region labels,120
3,Standardised contract_type labels,180
4,Converted blank total_charges to numeric,25
5,Imputed missing monthly_charges (plan median),95
6,Filled missing payment_method with 'Unknown',60


In [6]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
tenure_months,7500.0,24.73,17.44,1.00,11.00,23.00,37.00,72.00
internet_speed_mbps,7500.0,393.93,287.76,100.00,100.00,300.00,500.00,1000.00
monthly_charges,7500.0,85.43,27.85,27.29,59.44,85.77,100.77,153.39
support_tickets_90d,7500.0,1.20,1.10,0.00,0.00,1.00,2.00,6.00
outages_90d,7500.0,0.80,0.89,0.00,0.00,1.00,1.00,6.00
total_charges,7500.0,2126.16,1726.24,34.26,787.58,1792.92,3056.42,10595.52
churn_flag,7500.0,0.20,0.40,0.00,0.00,0.00,0.00,1.00


## 3. Segment analysis (SQL)
Queries live in `sql/churn_queries.sql` and run against SQLite here.

In [7]:
results = run_sql(df)
results['overall_kpis']

,customers,churned_customers,churn_rate_pct,monthly_revenue_lost,avg_monthly_charge
0,7500,1521,20.28,140554.24,85.43


In [8]:
results['churn_by_contract']

,contract_type,customers,churned,churn_rate_pct
0,Month-to-month,4178,1274,30.5
1,One year,1864,191,10.2
2,Two year,1458,56,3.8


In [9]:
results['churn_by_tenure_band']

,tenure_band,customers,churn_rate_pct
0,0-6 mo,1326,38.2
1,7-12 mo,840,29.5
2,13-24 mo,1853,23.1
3,25-48 mo,2684,11.9
4,49-72 mo,797,2.5


In [10]:
results['service_issues_impact']

,issue_bucket,customers,churn_rate_pct
0,0 issues,970,12.7
1,1-2 issues,4134,17.0
2,3-4 issues,2008,27.5
3,5+ issues,388,37.1


In [11]:
results['churn_by_region']

,region,customers,churn_rate_pct,churn_rank
0,South Texas,1296,22.8,1
1,North Texas,1215,20.6,2
2,Southeast,1299,20.5,3
3,Midwest,1241,20.0,4
4,West,1208,19.0,5
5,Northeast,1241,18.7,6


In [12]:
results['high_risk_segment']

,customers,churn_rate_pct,monthly_revenue_in_segment
0,243,56.8,20919.33


## 4. Explainable churn-risk model

In [13]:
model = fit_model(df)
print('Test AUC:', model['test_auc'])
pd.Series(model['coefficients']).sort_values()

Test AUC: 0.801


contract_type_Two year   -0.600
tenure_months            -0.599
contract_type_One year   -0.447
autopay                  -0.242
outages_90d               0.287
support_tickets_90d       0.326
monthly_charges           0.342
dtype: float64

![drivers](../reports/figures/churn_drivers.png)

See `reports/summary.md` for the plain-language write-up and recommendations.